# BeautyCo

In [10]:
pip install "xlrd>=2.0.1"

/usr/lib/python3.14/pty.py:66: DeprecationWarning: This process (pid=294687) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import itertools

import pandas as pd
from mlxtend.frequent_patterns import association_rules, fpgrowth

## 2. Data Preprocessing

In [12]:
df = pd.read_excel('BeautyCo_68.xls')
df.columns = [c.strip() for c in df.columns]
df = df.set_index('Trans.')

products = list(df.columns)
basket = df[products].astype(bool)

print('Transactions x products:', basket.shape)
print('Missing values:', int(df.isna().sum().sum()))
basket.head()

Transactions x products: (1000, 14)
Missing values: 0


,Travel_Pouch,Cheek_Color,Nail_Art,Makeup_Tools,Coverage_Base,Brow_Definer,Contouring,Lip_Liner,Eye_Enhancer,Eye_Palette,Base_Makeup,Glossy_Lips,Lip_Color,Eye_Liner
Trans.,,,,,,,,,,,,,,
1,False,True,True,True,True,False,True,True,True,False,False,False,False,True
2,False,False,True,False,True,False,True,True,False,False,True,True,False,False
3,False,True,False,False,True,True,True,True,True,True,True,True,True,False
4,False,False,True,True,True,False,True,False,False,False,True,False,False,True
5,False,True,False,False,True,False,True,True,True,True,False,True,True,False


## 3. EDA

In [ ]:
basket.mean().sort_values(ascending=False).round(3).to_frame('support')

Transactions : 1000
Products : 14
Items/basket : 4.39


,support
Base_Makeup,0.536
Glossy_Lips,0.490
Eye_Liner,0.457
Coverage_Base,0.442
Eye_Palette,0.381
Cheek_Color,0.363
Eye_Enhancer,0.357
Lip_Color,0.322
Nail_Art,0.280
Contouring,0.279


**Which products are most frequently purchased together?**

In [14]:
pairs = []
for a, b in itertools.combinations(products, 2):
    support = (basket[a] & basket[b]).mean()
    pairs.append({'A': a, 'B': b, 'support': support,
                  'lift': support / (basket[a].mean() * basket[b].mean())})

pair_stats = pd.DataFrame(pairs)
pair_stats.nlargest(5, 'support').round(3).reset_index(drop=True)

,A,B,support,lift
0,Base_Makeup,Glossy_Lips,0.356,1.355
1,Eye_Enhancer,Eye_Palette,0.321,2.360
2,Coverage_Base,Eye_Liner,0.297,1.470
3,Base_Makeup,Eye_Liner,0.238,0.972
4,Coverage_Base,Base_Makeup,0.231,0.975


**Are there any unexpected product combinations?**

In [15]:
pair_stats.nlargest(3, 'lift').round(3).reset_index(drop=True)

,A,B,support,lift
0,Nail_Art,Makeup_Tools,0.149,3.571
1,Travel_Pouch,Makeup_Tools,0.020,2.486
2,Eye_Enhancer,Eye_Palette,0.321,2.360


In [16]:
pair_stats.nsmallest(2, 'lift').round(3).reset_index(drop=True)

,A,B,support,lift
0,Lip_Liner,Base_Makeup,0.087,0.694
1,Brow_Definer,Glossy_Lips,0.016,0.777


Nail_Art + Makeup_Tools has the highest lift in the data but only 0.149 support,
so it is pruned before FP-Growth can use it. Lip_Liner + Base_Makeup (lift 0.69) is a negative
association — these two are avoided together.

## 4. Association Rule Modeling

In [17]:
frequent_itemsets = fpgrowth(basket, min_support=0.2, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values(['length', 'support'], ascending=[True, False])

print('Frequent itemsets:', len(frequent_itemsets))
print(frequent_itemsets['length'].value_counts().sort_index().to_string())

Frequent itemsets: 23
length
1    11
2    12


23 frequent itemsets - 11 single items and 12 pairs.

In [18]:
all_sets = list(frequent_itemsets['itemsets'])
maximal = [s for s in all_sets if not any(s < other for other in all_sets)]

print('Maximal frequent itemsets:', len(maximal))

(frequent_itemsets[frequent_itemsets['itemsets'].isin(maximal)]
 .assign(itemsets=lambda d: d['itemsets'].apply(lambda s: ', '.join(sorted(s))))
 .sort_values(['length', 'support'], ascending=False)
 .round(3).reset_index(drop=True))

Maximal frequent itemsets: 16


,support,itemsets,length
0,0.356,"Base_Makeup, Glossy_Lips",2
1,0.321,"Eye_Enhancer, Eye_Palette",2
2,0.297,"Coverage_Base, Eye_Liner",2
3,0.238,"Base_Makeup, Eye_Liner",2
4,0.231,"Base_Makeup, Coverage_Base",2
5,0.227,"Eye_Liner, Glossy_Lips",2
6,0.220,"Cheek_Color, Coverage_Base",2
7,0.218,"Coverage_Base, Glossy_Lips",2
8,0.211,"Base_Makeup, Eye_Palette",2
9,0.204,"Coverage_Base, Eye_Enhancer",2


16 maximal itemsets - all 12 pairs, plus 4 single items that pair with nothing at 0.2 support.

In [19]:
rules = (association_rules(frequent_itemsets.drop(columns='length'),
                          metric='confidence', min_threshold=0.7)
         .sort_values('confidence', ascending=False).reset_index(drop=True))

print('Rules with confidence >= 0.7:', len(rules))

rules.assign(rule=lambda d: d['antecedents'].apply(lambda s: ', '.join(sorted(s)))
                            + ' -> '
                            + d['consequents'].apply(lambda s: ', '.join(sorted(s))))[
    ['rule', 'support', 'confidence', 'lift']].round(3)

Rules with confidence >= 0.7: 3


,rule,support,confidence,lift
0,Eye_Enhancer -> Eye_Palette,0.321,0.899,2.360
1,Eye_Palette -> Eye_Enhancer,0.321,0.843,2.360
2,Glossy_Lips -> Base_Makeup,0.356,0.727,1.355


## 5. Deployment and Interpretation

In [20]:
best = rules.iloc[0]
a = basket[list(best['antecedents'])].all(axis=1)
c = basket[list(best['consequents'])].all(axis=1)

support = (a & c).mean()
confidence = (a & c).sum() / a.sum()
lift = confidence / c.mean()

print('Highest-confidence rule:',
      ', '.join(sorted(best['antecedents'])), '->', ', '.join(sorted(best['consequents'])))
print()
print(f'Support    = {(a & c).sum()}/{len(basket)} = {support:.3f}')
print(f'Confidence = {(a & c).sum()}/{a.sum()} = {confidence:.3f}')
print(f'Lift       = {confidence:.3f}/{c.mean():.3f} = {lift:.3f}')

Highest-confidence rule: Eye_Enhancer -> Eye_Palette

Support    = 321/1000 = 0.321
Confidence = 321/357 = 0.899
Lift       = 0.899/0.381 = 2.360


If a customer buys Eye_Enhancer, then they also buy Eye_Palette.

Support 0.321, confidence 0.899, lift 2.36. Only 38.1% of all customers buy a palette, but among
mascara buyers it is 89.9% - 2.36× more likely than average.

Marketing use: sell the two as an "Eye Look Set" bundle